This notebook is for building our own lightweight version of Claimify. Claimify uses a 4-stage pipeline:
1. Sentence Splitting: Breaks text into individual sentences with surrounding context
2. Selection: Filters for sentences containing verifiable propositions, excluding opinions and speculation
3. Disambiguation: Resolves ambiguities or discards sentences that cannot be clarified
4. Decomposition: Breaks down sentences into atomic, self-contained factual claims

In [1]:
import nltk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from fastcoref.modeling import FCoref, FCorefModel
import torch

In [2]:
#Download the sentence tokenizer for sentence splitting
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
#Initialize the fastcoref model for disambiguation
if not hasattr(FCorefModel, "all_tied_weights_keys"):
    FCorefModel.all_tied_weights_keys = {}

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
coref_model = FCoref(device=device)

04/06/2026 11:46:08 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:46:08 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/06/2026 11:46:08 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:46:08 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/06/2026 11:46:08 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:46:08 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/tokenizer_config.json

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

FCorefModel LOAD REPORT from: biu-nlp/f-coref
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
04/06/2026 11:46:08 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref "HTTP/1.1 200 OK"
04/06/2026 11:46:08 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/main "HTTP/1.1 200 OK"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/discussions?p=0 "HTTP/1.1 200 OK"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Fou

In [4]:
#Initialize a lightweight zero-shot classification model for selection
claim_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3",
    device=-1 # Set to 0 if using a GPU
)

04/06/2026 11:46:09 - INFO - 	 HTTP Request: HEAD https://huggingface.co/valhalla/distilbart-mnli-12-3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/valhalla/distilbart-mnli-12-3/ef9a58ce6a9cd44cd0d4c2f7db1cd67f81019a8b/config.json "HTTP/1.1 200 OK"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: HEAD https://huggingface.co/valhalla/distilbart-mnli-12-3/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3 "HTTP/1.1 200 OK"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/commits/main "HTTP/1.1 200 OK"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/discussions?p=0 "HTTP/1.1 200 OK"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: GET https://h

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

04/06/2026 11:46:09 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04/06/2026 11:46:09 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


In [6]:
#Load FLAN-T5 for text rewriting/decomposition
#Wraps the model and tokenizer so it can accept string inputs
decompose_model = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=-1 # Use device=0 if you are running this on a GPU
)

04/06/2026 11:51:48 - INFO - 	 HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:51:48 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/config.json "HTTP/1.1 200 OK"
04/06/2026 11:51:49 - INFO - 	 HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:51:49 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

04/06/2026 11:51:49 - INFO - 	 HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:51:49 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/generation_config.json "HTTP/1.1 200 OK"
04/06/2026 11:51:49 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04/06/2026 11:51:49 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDeco

In [7]:
def resolve_coreferences(text):
    """
    Replaces pronouns and implicit references in the text with their explicit entities.
    """
    #Predict coreference clusters
    preds = coref_model.predict(texts=[text])

    #Get clusters as character start/end indices
    clusters = preds[0].get_clusters(as_strings=False)

    replacements = []
    for cluster in clusters:
        #The first mention in a cluster is usually the explicit entity (the antecedent)
        primary_start, primary_end = cluster[0]
        primary_text = text[primary_start:primary_end]

        # We want to replace all subsequent mentions (usually pronouns) with the primary text
        for mention_start, mention_end in cluster[1:]:
            replacements.append((mention_start, mention_end, primary_text))

    #Sort replacements in reverse order of their start index
    replacements.sort(key=lambda x: x[0], reverse=True)

    #Apply replacements
    resolved_text = text
    for start, end, rep_text in replacements:
        resolved_text = resolved_text[:start] + rep_text + resolved_text[end:]

    return resolved_text

In [12]:
def decompose_sentence(sentence):
    """
    Uses a small instruction-tuned model to break a complex sentence
    into simple, atomic claims.
    """
    prompt = f"Decompose the following sentence into simple, independent facts. Separate each fact with a pipe '|' character. Sentence: {sentence}"

    #1. Use max_new_tokens instead of max_length
    #2. Add return_full_text=False so we only get the answer, not the prompt
    output = decompose_model(
        prompt,
        max_new_tokens=128,
        truncation=True,
        return_full_text=False
    )[0]['generated_text']

    #Fallback: Just in case the model STILL spits out the prompt, strip it manually
    if output.startswith(prompt):
        output = output[len(prompt):].strip()

    #Split the output by our pipe character and clean it up
    atomic_claims = [claim.strip() for claim in output.split('|') if len(claim.strip()) > 5]

    if not atomic_claims:
        return [sentence]

    return atomic_claims

In [13]:
def lightweight_claimify(text, threshold=0.6):
    """
    The complete, keyless 4-stage claim extraction pipeline.
    """
    #Stage 1: Disambiguation (Coreference Resolution)—we're doing first instead of third because it works best with our lightweight, API free approach
    disambiguated_text = resolve_coreferences(text)

    #Stage 2: Sentence Splitting
    complex_sentences = nltk.sent_tokenize(disambiguated_text)
    atomic_sentences = []

    #Stage 3: Decomposition
    for sent in complex_sentences:
        #Only try to decompose longer sentences with conjunctions
        if " and " in sent.lower() or " but " in sent.lower() or "," in sent:
            decomposed = decompose_sentence(sent)
            atomic_sentences.extend(decomposed)
        else:
            atomic_sentences.append(sent)

    extracted_claims = []

    #Stage 4: Selection (Filtering opinions vs. facts)
    for claim in atomic_sentences:
        if len(claim.split()) < 4:
            continue

        result = claim_classifier(
            claim,
            candidate_labels=["factual claim", "personal opinion"],
            multi_label=False
        )

        #Keep only the confident factual claims
        if result['labels'][0] == "factual claim" and result['scores'][0] >= threshold:
            extracted_claims.append(claim)

    #Clean up duplicates that can happen during decomposition
    return list(set(extracted_claims))

In [14]:
#Testing
sample_article = """
I think the new policies are an absolute disaster.
The inflation rate in the country rose by 4.2% last quarter.
It is the worst economic decision in history!
According to the report, the city council voted 5-2 to pass the infrastructure bill.
This bill will go into effect next year and raise taxes 5%.
"""

In [15]:
print("Original Text:")
print(sample_article.strip())
print("\n--- Extracted Verifiable Claims ---")
claims = lightweight_claimify(sample_article)
for claim in claims:
    print(f"- {claim}")

04/06/2026 11:54:12 - INFO - 	 Tokenize 1 inputs...


Original Text:
I think the new policies are an absolute disaster.
The inflation rate in the country rose by 4.2% last quarter.
It is the worst economic decision in history!
According to the report, the city council voted 5-2 to pass the infrastructure bill.
This bill will go into effect next year and raise taxes 5%.

--- Extracted Verifiable Claims ---


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/06/2026 11:54:13 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

- According to the report, the city council voted 5-2 to pass the infrastructure bill.
- the infrastructure bill will go into effect next year and raise taxes 5%.
- The inflation rate in the country rose by 4.2% last quarter.
